In [70]:
# Import packages
from typing import Optional
import phenograph
import scanpy as sc
from anndata import AnnData
from anndata import concat
import seaborn as sns
import matplotlib.pyplot as plt
import pathlib
import os
import pandas as pd
import numpy as np
from scipy.stats import entropy
import subprocess
import palantir
import sys
import random
from collections import OrderedDict
import re
from itertools import chain
import warnings
sys.path.append('/lila/home/forsythb/anaconda3/envs/scrna/lib/python3.8/site-packages')
import harmony
from scipy.sparse import coo_matrix
from sklearn.neighbors import NearestNeighbors
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
%matplotlib inline

In [71]:
# This is the full patient 146 dataset
adata_patient = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Patients.HTAN/adatas/KG146_Patient_Organoid.h5ad')
print(adata_patient.shape)

(12016, 20157)


In [72]:
# Subset the patient 146 dataset to include only the labels we want
adata_patient = adata_patient[~((adata_patient.obs['Cell State'] == 'NA') | (adata_patient.obs['Cell State'] == 'nan') | (adata_patient.obs['Cell State'] == 'Organoid'))]

In [73]:
adata_patient.obs['Cell State']

120703424285939_KG146M            ISC-like
120703436155741_KG146M                 SCC
120703455025013_KG146M            ISC-like
120718456679846_KG146M    Fetal Progenitor
120718468987109_KG146M     Enterocyte-like
                                ...       
241109220584862_KG146M                 SCC
241176048225691_KG146M            ISC-like
241176061270774_KG146M         Goblet-like
241184516782309_KG146M                 SCC
241184517057252_KG146M             TA-like
Name: Cell State, Length: 1304, dtype: category
Categories (8, object): ['Early NET', 'Enterocyte-like', 'Fetal Progenitor', 'Goblet-like', 'ISC-like', 'Injury Repair', 'SCC', 'TA-like']

In [74]:
# Perform DEGs with wilcoxon
sc.tl.rank_genes_groups(adata_patient, groupby='Cell State', method='wilcoxon', key_added = "wilcoxon", n_genes=100)
#sc.pl.rank_genes_groups(adata_patient, n_genes=100, sharey=False, key="wilcoxon")

results_dict = adata_patient.uns['wilcoxon']

# Create a dictionary to store dataframes for each group
deg_dfs = {}

# Iterate over each group and create a dataframe
for group in results_dict['names'].dtype.names:
    df = pd.DataFrame(results_dict['names'][group], columns=['gene'])
    df['logfoldchanges'] = results_dict['logfoldchanges'][group]
    df['pvals'] = results_dict['pvals'][group]
    df['pvals_adj'] = results_dict['pvals_adj'][group]
    
    # Filter out highly significant genes
    significant_genes = df[(df['logfoldchanges'].abs() > 3) & (df['pvals_adj'] < 0.001)]
    
    # Store the filtered dataframe in the dictionary
    deg_dfs[group] = significant_genes
    
# Concatenate significant genes from different groups into a single dataframe
concatenated_data = pd.concat([deg_dfs[group] for group in deg_dfs], axis=0)

/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:582: ImplicitModificationWarning: Trying to modify attribute `._uns` of view, initializing view as actual.
  adata.uns[key_added] = {}


/home/forsythb/.local/lib/python3.9/site-packages/numpy/core/fromnumeric.py:86: FutureWarning: The behavior of DataFrame.sum with axis=None is deprecated, in a future version this will reduce over both axes and return a scalar. To retain the old behavior, pass axis=0 (or do not pass axis)
  return reduction(axis=axis, out=out, **passkwargs)
/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:419: RuntimeWarning: overflow encountered in expm1
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (
/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:420: RuntimeWarning: overflow encountered in expm1
  self.expm1_func(mean_rest) + 1e-9
/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:419: RuntimeWarning: invalid value encountered in divide
  foldchanges = (self.expm1_func(mean_group) + 1e-9) / (
/home/forsythb/.local/lib/python3.9/site-packages/scanpy/tools/_rank_genes_groups.py:419: Run

In [76]:
len(concatenated_data)

619

In [80]:
# Get the list of significant genes
significant_genes_list = concatenated_data['gene'].tolist()
len(significant_genes_list)

# Filter the AnnData object to contain only the significant genes
adata_patient_filtered = adata_patient[:, adata_patient.var_names.isin(significant_genes_list)]

In [82]:
# This is the organoid dataset
adata_organoid = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/postprocess_adata.020624/adata.combined.postprocess.h5ad')

In [87]:
# Ensure both objects have the same set of genes
common_genes = adata_patient_filtered.var_names.intersection(adata_organoid.var_names)

# Subset both objects to include only common genes
adata_patient_filtered_common = adata_patient_filtered[:, common_genes].copy()
adata_organoid_common = adata_organoid[:, common_genes].copy()

In [90]:
# Create a 'Cell State' column where we fill in the values as NA
adata_organoid_common.obs['Cell State'] = pd.NA

In [91]:
# Subset the organoid data
# Base media, liver metastasis
adata_base_liver = adata_organoid_common[(adata_organoid_common.obs['Culture_Media'] == 'BASE') & 
                                  (adata_organoid_common.obs['Tumor_Site'] == 'Metastatic')]
# Base media, primary 
adata_base_primary = adata_organoid_common[(adata_organoid_common.obs['Culture_Media'] == 'BASE') & 
                                  (adata_organoid_common.obs['Tumor_Site'] == 'Primary')]

# HISC media, liver metastasis
adata_hisc_liver = adata_organoid_common[(adata_organoid_common.obs['Culture_Media'] == 'HISC') & 
                                  (adata_organoid_common.obs['Tumor_Site'] == 'Metastatic')]
# HISC media, primary
adata_hisc_primary = adata_organoid_common[(adata_organoid_common.obs['Culture_Media'] == 'HISC') & 
                                  (adata_organoid_common.obs['Tumor_Site'] == 'Primary')]

# Dedifferentiated media, liver metastasis
adata_dedif_liver = adata_organoid_common[(adata_organoid_common.obs['Culture_Media'] == 'Dedifferentiated') & 
                                  (adata_organoid_common.obs['Tumor_Site'] == 'Metastatic')]
# Dedifferentiated media, primary
adata_dedif_primary = adata_organoid_common[(adata_organoid_common.obs['Culture_Media'] == 'Dedifferentiated') & 
                                  (adata_organoid_common.obs['Tumor_Site'] == 'Primary')]

### Rerun preprocessing for all data

In [120]:
# Concatenate the data using anndata.concat
concatenated_data = concat([adata_patient_filtered_common, adata_hisc_primary])

# Extract count matrix
counts_patient_organoid = pd.DataFrame(concatenated_data.X.toarray(),
                                       index=concatenated_data.obs_names,
                                       columns=concatenated_data.var_names)

/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/merge.py:1263: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(


In [121]:
# Reprocess the data
# 1. Normalization
norm_df = harmony.utils.normalize_counts(counts_patient_organoid)

# Gene selection
hvg_genes = harmony.utils.hvg_genes(norm_df)

# Log transform
data_df = harmony.utils.log_transform(norm_df.loc[:,hvg_genes])

/lila/home/forsythb/anaconda3/envs/scrna/lib/python3.8/site-packages/harmony/utils.py:81: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  disp_grouped = df.groupby('mean_bin')['dispersion']


In [122]:
data_df.head()
data_df.tail()

,DEFA6,GNE,ADAMTS18,CADPS,PTPRR,PTPRN2,ITPR2,DKK4,FOXP1,GMDS,...,TUBA1C,CDKN1C,NECTIN4,SSTR2,TPD52,DLGAP5,ANXA1,EMP1,CDKN3,MUC15
146P_HISC_shCTRL_GCTACAACACGATTCA-1,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,5.577302,-3.321928,6.240635,7.064567,...,6.240635,-3.321928,-3.321928,-3.321928,5.577302,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
146P_HISC_shZFP36L2_4_GAGTGTTTCCTGTAAG-1,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,5.487590,-3.321928,6.150852,5.487590,...,6.701082,-3.321928,-3.321928,-3.321928,6.485981,5.487590,-3.321928,-3.321928,-3.321928,-3.321928
146P_HISC_shZFP36L2_4_CCTCCTCCATATGGCT-1,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,6.910491,-3.321928,6.910491,6.540698,...,-3.321928,-3.321928,-3.321928,-3.321928,6.540698,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928
146P_HISC_shZFP36L2_4_ATGTCTTCAGTCCCGA-1,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,5.373974,6.372233,-3.321928,6.860944,6.741982,...,5.373974,-3.321928,-3.321928,-3.321928,6.037138,-3.321928,-3.321928,-3.321928,5.373974,-3.321928
146P_HISC_shCTRL_CCTCACAGTTGGTGTT-1,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928,6.244251,-3.321928,5.246153,6.828579,...,-3.321928,-3.321928,-3.321928,-3.321928,5.246153,-3.321928,-3.321928,-3.321928,-3.321928,-3.321928


## Get Affinities

In [123]:
# The patient data is defined as those cells with 'KG146M' in the row name
# Define the time point connections
tp = pd.Series(index=data_df.index)

# Define pattern for Patient cells
patient_pattern = '_KG146'

# Use str.contains to check if the patient pattern is present in the cell identifier
patient_cells = data_df.index[data_df.index.str.contains(patient_pattern)]

# Assign 'Patient' to the corresponding cells in the series
tp[patient_cells] = 'Patient'

# The remaining cells (not classified as Patient) are Organoid cells
tp.fillna('Organoid', inplace=True)

organoid_count = (tp == 'Organoid').sum()
patient_count = (tp == 'Patient').sum()

print(f"Number of Organoid cells: {organoid_count}")
print(f"Number of Patient cells: {patient_count}")

Number of Organoid cells: 9688
Number of Patient cells: 1304


/scratch/lsftmp/3573472.tmpdir/ipykernel_117805/4160515974.py:12: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value 'Patient' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  tp[patient_cells] = 'Patient'


In [124]:
# Define timepoint connections
timepoint_connections = pd.DataFrame(columns=[0, 1])
index = 0
timepoint_connections.loc[index, :] = ['Organoid', 'Patient']; index += 1
timepoint_connections

,0,1
0,Organoid,Patient


In [125]:
'''
Harmony and Palantir
'''
def _mnn_ka_distances(mnn, n_neighbors):
    # Function to find distance ka^th neighbor in the mutual nearest neighbor matrix
    ka = int(n_neighbors / 3)
    ka_dists = np.repeat(None, mnn.shape[0])
    x, y, z = find(mnn)
    rows=pd.Series(x).value_counts()
    for r in rows.index[rows >= ka]:
        ka_dists[r] = np.sort(z[x==r])[ka - 1]
    return ka_dists

#######################################################################################
from harmony import core
from sklearn.neighbors import NearestNeighbors
from scipy.sparse import find, csr_matrix

def harmony_aug_mat_with_pca(projections, timepoints, timepoint_connections):
    # Time point cells and indices
    tp_cells = pd.Series()
    tp_offset = pd.Series()
    offset = 0
    for i in timepoints.unique():
        tp_offset[i] = offset
        tp_cells[i] = list(timepoints.index[timepoints == i])
        offset += len(tp_cells[i])
    n_neighbors = 30
    
    # Nearest neighbor graph construction and affinity matrix
    print('Nearest neighbor computation...')
    nbrs = NearestNeighbors(n_neighbors=n_neighbors,
                            metric='cosine', n_jobs=-2).fit(projections.values)

    adj = nbrs.kneighbors_graph(projections.values, mode='distance')
    dists, _ = nbrs.kneighbors(projections.values)
    
    # Scaling factors for affinity matrix construction
    ka = int(n_neighbors / 3)
    scaling_factors = pd.Series(dists[:, ka], index=projections.index)
    
    # Affinity matrix
    nn_aff = core._convert_to_affinity(adj, scaling_factors, True)
    n_jobs = -2
    
    # Mututally nearest neighbor affinity matrix
    # Initilze mnn affinity matrix
    N = projections.shape[0]
    full_mnn_aff = csr_matrix(([0], ([0], [0])), [N, N])
    for i in timepoint_connections.index:
        t1, t2 = timepoint_connections.loc[i, :].values
        print(f'Constucting affinities between {t1} and {t2}...')
        # MNN matrix  and distance to ka the distance
        t1_cells = tp_cells[t1]
        t2_cells = tp_cells[t2]
        mnn = core._construct_mnn(t1_cells, t2_cells, projections,
                             n_neighbors, n_jobs)
        
        # MNN Scaling factors
        # Distance to the adaptive neighbor
        ka_dists = pd.Series(0.0, index=t1_cells + t2_cells)
        ka_dists = ka_dists.astype(float)
        # T1 scaling factors
        ka_dists[t1_cells] = _mnn_ka_distances(mnn, n_neighbors)
        # T2 scaling factors
        ka_dists[t2_cells] = _mnn_ka_distances(mnn.T, n_neighbors)
        # Scaling factors
        mnn_scaling_factors = pd.Series(0.0, index=projections.index)
        mnn_scaling_factors[t1_cells] = core._mnn_scaling_factors(
            ka_dists[t1_cells], scaling_factors)
        mnn_scaling_factors[t2_cells] = core._mnn_scaling_factors(
            ka_dists[t2_cells], scaling_factors)
        # MNN affinity matrix
        full_mnn_aff = full_mnn_aff + \
            core._mnn_affinity(mnn, mnn_scaling_factors,
                          tp_offset[t1], tp_offset[t2])
    # Symmetrize the affinity matrix and return
    aug_aff2 = nn_aff + nn_aff.T + full_mnn_aff + full_mnn_aff.T
    aff2 = nn_aff + nn_aff.T
    return aug_aff2, aff2

In [126]:
pca_merge = pd.DataFrame(concatenated_data.obsm['X_pca'], index=concatenated_data.obs_names)
aug_mat, aff_mat = harmony_aug_mat_with_pca(pca_merge, tp, timepoint_connections)

Nearest neighbor computation...
Constucting affinities between Organoid and Patient...
t+1 neighbors of t...
t neighbors of t+1...


/scratch/lsftmp/3573472.tmpdir/ipykernel_117805/3931875684.py:64: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise in a future error of pandas. Value '[None None None ... None 17.74883403528355 None]' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  ka_dists[t1_cells] = _mnn_ka_distances(mnn, n_neighbors)


In [127]:
# Save the combined adata object
#combined_adata.write_h5ad(os.path.join(out_dir, 'combined_adata.h5'))
out_dir = '/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/harmony_phenograph/harmony_phenograph.021524/'
os.makedirs(out_dir,exist_ok=True)

In [128]:
# Convert arrays to string format
# Convert non-string columns in the "obs" DataFrame to strings
concatenated_data.obs = concatenated_data.obs.applymap(str)

# Add aug_mat to adata_patient_organoid
concatenated_data.obsm['aug_mat'] = aug_mat.toarray()

# Add aff_mat to adata_patient_organoid
concatenated_data.obsm['aff_mat'] = aff_mat.toarray()

# Save the modified patient 146 organoid adata 
concatenated_data.write('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/harmony_phenograph/harmony_phenograph.021524/adata_harmony_hisc_primary.h5ad')

/scratch/lsftmp/3573472.tmpdir/ipykernel_117805/2142335431.py:3: FutureWarning: DataFrame.applymap has been deprecated. Use DataFrame.map instead.
  concatenated_data.obs = concatenated_data.obs.applymap(str)


## PhenoGraph Classification

In [129]:
adata_patient_organoid = sc.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/harmony_phenograph/harmony_phenograph.021524/adata_harmony_hisc_primary.h5ad')

In [130]:
aug_mat = pd.DataFrame(adata_patient_organoid.obsm['aug_mat'], index=adata_patient_organoid.obs.index, columns = adata_patient_organoid.obs.index)
adata_patient_organoid.obs['Cell State'] = adata_patient_organoid.obs['Cell State'].astype(str).fillna('NaN')

In [131]:
ind = adata_patient_organoid.obs['Cell State'] != 'NaN'
adata_patient_organoid = adata_patient_organoid[ind, :]
aug_mat = aug_mat.loc[ind, ind]

In [132]:
# Select cells with labeled cell state (excluding 'NA')
ind = adata_patient_organoid.obs['Cell State'] != 'nan'
labeled_cells = adata_patient_organoid[ind,:]

# Get unique cell types/categories
cell_types = labeled_cells.obs['Cell State'].unique()

# Initialize an empty list to store training data for each cell type
train_data = []
train_labels = []
ct_codes = []

# Iterate over cell types and extract 'aug_mat' for each
for i,cell_type in enumerate(cell_types):
    # Filter cells of the current cell type
    cells_of_type = labeled_cells[labeled_cells.obs['Cell State'] == cell_type]
    
    # Extract 'aug_mat' for these cells
    pc_data = cells_of_type.obsm['X_pca']  # Assuming sparse matrix to array
    
    # Append 'aug_mat' data to the list
    train_data.append(pc_data)
    train_labels.append(cells_of_type.obs.index.to_series())   
    ct_codes += [i+1]*pc_data.shape[0]

In [133]:
train_labels = pd.concat(train_labels).index

In [134]:
# # Select cells with labeled cell state (excluding 'nan') for test data
test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'] == 'nan']
#adata_patient_organoid.obs['Cell State'] = adata_patient_organoid.obs['Cell State'].replace('NaN', np.nan)
#test_cells = adata_patient_organoid[adata_patient_organoid.obs['Cell State'].isna()]

# # Extract 'aug_mat' for test cells
test_data = test_cells.obsm['X_pca']
test_labels = test_cells.obs.index

In [135]:
labels = pd.Index(test_labels.tolist() +  train_labels.tolist())
aug_all = coo_matrix(aug_mat.loc[labels,labels].values )
ct_codes = np.array([0]*len(test_labels) + ct_codes)

In [136]:
def preprocess(train, test):
    labels = np.zeros((test.shape[0], ), dtype=int)
    data = test
    for c, examples in enumerate(train):
        labels = np.append(labels, np.tile(c+1, (examples.shape[0], )), axis=0)
        data = np.append(data, examples, axis=0)
    # Check that results are valid
    if labels[-1] == 0:
        raise IndexError("Last entry in labels should not be 0")
    if labels.shape[0] != data.shape[0]:
        raise IndexError("Data and labels should be the same length")
    if sum(labels == 0) != test.shape[0]:
        raise IndexError("Labels should include one 0 for every row of test data")
    return data, labels

In [137]:
data, labels = preprocess(train_data, test_data)

In [138]:
def create_graph(data, k=30, metric='euclidean', n_jobs=-1):
    # def _kernel(dxy, sigma=1):
    #     return np.exp(-dxy ** 2 / sigma)

    _, idx = find_neighbors(data, k=k, metric=metric, n_jobs=n_jobs)
    # affinities = np.apply_along_axis(lambda x: _kernel(x, x.std()), axis=1, arr=d)
    # n, k = idx.shape
    # i = [np.tile(x, (k, )) for x in range(n)]
    # i = np.concatenate(np.array(i))
    # j = np.concatenate(idx)
    # s = np.concatenate(affinities)
    # graph = sp.coo_matrix((s, (i, j)), shape=(n, n)).tocsr()
    # graph = (graph + graph.transpose()).multiply(.5)
    graph = neighbor_graph(jaccard_kernel, {'idx': idx})
    # make symmetric
    # graph = (graph + graph.transpose()).multiply(.5)
    return graph

In [139]:
# Assuming aug_all is your coo_matrix
subset_rows = slice(0, 50)
subset_cols = slice(0, 10)
subset_aug_all = aug_all.tocsr()[subset_rows, subset_cols]

# Now pass the subset to the create_graph function
# create_graph(subset_aug_all)

# dense_subset_aug_all = subset_aug_all.toarray()
# create_graph(dense_subset_aug_all)

In [140]:
dense_subset_aug_all = subset_aug_all.toarray()
create_graph(dense_subset_aug_all)

Finding 30 nearest neighbors using minkowski metric and 'auto' algorithm


<50x50 sparse matrix of type '<class 'numpy.float64'>'
	with 1500 stored elements in COOrdinate format>

In [141]:
import numpy as np
from phenograph.core import find_neighbors, neighbor_graph, jaccard_kernel
import scipy.sparse as sp
from sklearn.preprocessing import normalize


def random_walk_probabilities(A, labels):

    if labels.min() != 0:
        raise ValueError("Labels should encode unlabeled points with 0s")

    # Check if input is sparse
    if sp.issparse(A):
        if not isinstance(A, sp.csr_matrix):
            A = A.tocsr()
        D = sp.diags(A.sum(axis=1).flatten(), [0], shape=A.shape, format='csr')
        L = D - A
        seeds = labels != 0
        Lu = L[np.invert(seeds), :]  # unlabeled rows
        Lu = Lu.tocsc()[:, np.invert(seeds)]  # unlabeled columns
        # Check that Lu has the right size
        if not all([t == sum(np.invert(seeds)) for t in Lu.shape]):
            raise IndexError("Lu should be square and match size of test data")
        BT = L[np.invert(seeds), :]  # unlabeled rows
        BT = BT.tocsc()[:, seeds]  # labeled columns
        if not (sum(np.invert(seeds)), sum(seeds)) == BT.shape:
            raise IndexError("BT size is incorrect")
        # Matrix representation of labels
        i, j, s = [], [], []
        classes = np.unique(labels[seeds.nonzero()[0]])
        for k in classes:
            i.extend(np.where(labels[seeds] == k)[0])
            j.extend(np.tile(k, sum(seeds == k)))
            s.extend(np.tile(1, sum(seeds == k)))
        i = np.arange(seeds.sum())
        j = labels[seeds] - 1
        s = np.ones((seeds.sum(), ))
        M = sp.coo_matrix((s, (i, j)), shape=(seeds.sum(), len(classes))).tocsc()
        # P = sp.linalg.spsolve(Lu.tocsc(), -BT.dot(M))
        # Use iterative solver
        B = -BT.dot(M)
        vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]
        warnings = [x[1] for x in vals]
        if any(warnings):
            print("Warning: iterative solver failed to converge in at least one case", flush=True)
        P = normalize(np.vstack(tuple((x[0] for x in vals))).T, norm='l1')

    else:
        D = np.diag(np.sum(A, axis=1))
        L = D - A  # graph laplacian
        seeds = np.array([e != 0 for e in labels], dtype=bool)
        Lu = L[seeds, :][:, seeds]  # labeled rows, labeled cols
        BT = L[~seeds, :][:, seeds]  # unlabeled rows, labeled cols
        classes = np.unique(labels[labels > 0])
        M = np.zeros((seeds.sum(), len(classes)))
        for k in classes:
            M[labels[seeds] == k, k] = 1
        P = np.linalg.lstsq(Lu, np.dot(-BT, M))[0]

    return P

In [142]:
P = random_walk_probabilities(aug_all, ct_codes)

/scratch/lsftmp/3573472.tmpdir/ipykernel_117805/3655095487.py:42: DeprecationWarning: Please use `bicgstab` from the `scipy.sparse.linalg` namespace, the `scipy.sparse.linalg.isolve` namespace is deprecated.
  vals = [sp.linalg.isolve.bicgstab(Lu, b.T.todense()) for b in B.T]


In [143]:
pval_df = pd.DataFrame(P, index = test_cells.obs_names, columns = cell_types)

In [144]:
pval_df

,ISC-like,SCC,Fetal Progenitor,Enterocyte-like,TA-like,Injury Repair,Early NET,Goblet-like
146P_HISC_shZFP36L2_4_GTGCAGCAGAGAGGGC-1,0.256830,0.075374,0.048427,0.154813,0.157126,0.252274,0.016754,0.038403
146P_HISC_shCTRL_TGCAGATAGTCTACCA-1,0.141261,0.017753,0.010613,0.499473,0.034929,0.108700,0.006471,0.180799
146P_HISC_shCTRL_AAGAACAAGGACACTG-1,0.171064,0.019938,0.011903,0.438628,0.038472,0.108529,0.007090,0.204376
146P_HISC_shCTRL_GAATAGACAGGGATAC-1,0.195064,0.048162,0.029093,0.233121,0.183593,0.197025,0.020580,0.093361
146P_HISC_shZFP36L2_4_ACATCCCTCGCCACTT-1,0.317866,0.056968,0.042492,0.225330,0.060235,0.263923,0.010282,0.022904
...,...,...,...,...,...,...,...,...
146P_HISC_shCTRL_GCTACAACACGATTCA-1,0.168538,0.019893,0.011448,0.471683,0.041456,0.106862,0.007145,0.172975
146P_HISC_shZFP36L2_4_GAGTGTTTCCTGTAAG-1,0.349714,0.051726,0.038117,0.249109,0.063265,0.213838,0.010127,0.024105
146P_HISC_shZFP36L2_4_CCTCCTCCATATGGCT-1,0.221319,0.088568,0.060674,0.138356,0.164633,0.265061,0.019535,0.041853
146P_HISC_shZFP36L2_4_ATGTCTTCAGTCCCGA-1,0.435071,0.042060,0.032066,0.221536,0.055520,0.185873,0.008466,0.019407


In [145]:
pval_df.to_csv('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/harmony_phenograph/harmony_phenograph.021524/adata_phenograph_hisc_primary.csv', index=True)

In [151]:
import anndata

# Load the AnnData object from the h5ad file
adata = anndata.read_h5ad('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/postprocess_adata/postprocess_adata.020524/adata.combined.postprocess.h5ad')

# Extract the matrix
matrix = adata.X.toarray()

In [152]:
# Extract and save the obs DataFrame
obs_df = adata.obs
obs_df.to_csv('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/limma/obs.csv')

# Extract and save the var DataFrame
var_df = adata.var
var_df.to_csv('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/limma/var.csv')

In [153]:
# Optionally, you can also save the matrix as a separate file (e.g., a CSV or a NumPy array)
# Save the matrix as a CSV file
import pandas as pd
matrix_df = pd.DataFrame(matrix, index=adata.obs_names, columns=adata.var_names)
matrix_df.to_csv('/data/chanjlab/CRC_ZFP36L2.092023/Organoid/output_new/matrix.csv')

In [48]:
# Ensure both datasets have the same set of genes
common_genes = adata_patient.var_names.intersection(adata_organoid.var_names)
len(common_genes)

0

In [51]:
# Subset both datasets to include only common genes
adata_patient = adata_patient[:, adata_patient.var_names.isin(common_genes)].copy()
adata_patient.var_names_make_unique()
adata_organoid = adata_organoid[:, adata_organoid.var_names.isin(common_genes)].copy()
adata_organoid.var_names_make_unique()

/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:949: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    []

    Inferred to be: empty

  names = self._prep_dim_index(names, "var")
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:949: UserWarning: 
AnnData expects .var.index to contain strings, but got values like:
    []

    Inferred to be: empty

  names = self._prep_dim_index(names, "var")


In [63]:
# Drop duplicate indices in adata_patient and adata_base_primary
adata_patient = adata_patient[~adata_patient.obs.index.duplicated(keep='first')]
adata_base_primary = adata_base_primary[~adata_base_primary.obs.index.duplicated(keep='first')]

# Concatenate the data using anndata.concat
concatenated_data = concat([adata_patient, adata_organoid])

# Extract count matrix
counts_patient_organoid = pd.DataFrame(concatenated_data.X.toarray(),
                                       index=concatenated_data.obs_names,
                                       columns=concatenated_data.var_names)


/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/merge.py:1263: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  concat_annot = pd.concat(
/home/forsythb/.local/lib/python3.9/site-packages/anndata/_core/anndata.py:183: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [68]:
# Reprocess the data
# 1. Normalization
norm_df = harmony.utils.normalize_counts(counts_patient_organoid)

# Gene selection
hvg_genes = harmony.utils.hvg_genes(norm_df)

# Log transform
#data_df = harmony.utils.log_transform(norm_df.loc[:,hvg_genes])

IndexError: index -1 is out of bounds for axis 0 with size 0

In [69]:
counts_patient_organoid

""
120703424285939_KG146M
120703436155741_KG146M
120703455025013_KG146M
120718456679846_KG146M
120718468987109_KG146M
...
146P_HISC_shCTRL_CCTCACAGTTGGTGTT-1
146P_BASE_shZFP36L2_3_TCACGCTTCATAGGCT-1
KG146Li_BASE_shZFP36L2_3_TTCTCTCCACCTGCGA-1
146P_dedifferentiation_shZFP36L2_4_TCGGGTGCAAACGGCA-1
